# 난임 시술 임신 성공 예측 - 제출 파일 생성

main.py의 파이프라인(구조적 플래그 + 결측치 처리 + 죽은 컬럼 제거 + LightGBM 5-Fold 앙상블)을
그대로 따르며, 실험할 때마다 버전 관리되는 이름으로 제출 파일을 생성합니다.

**파일명 규칙**: `{N차}_{모델이름}_submit_improved({로컬점수}->{실제점수}).csv`
- 같은 차수 내에서는 실험을 몇 번 반복해도 파일명의 N차가 바뀌지 않습니다.
- 실제 대회에 제출하고 점수를 받은 뒤, 맨 아래 "제출 완료 표시" 셀을 실행하면
  다음 실험부터 차수가 1 올라갑니다.


## 0. 버전(차수) 관리

In [1]:
import json
import os

VERSION_FILE = "version_state.json"

def load_version_state():
    """현재 차수와 제출 여부를 파일에서 읽어온다. 없으면 1차로 새로 시작."""
    if os.path.exists(VERSION_FILE):
        with open(VERSION_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"version": 1, "submitted": False}

def save_version_state(state):
    with open(VERSION_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

def get_active_version():
    """
    제출 파일을 새로 만들 때 사용할 차수를 결정한다.

    이전에 제출 완료(submitted=True) 상태였다면 차수를 1 올리고
    submitted를 False로 리셋한 뒤 그 새 차수를 반환한다.
    아직 제출 전(submitted=False)이라면 같은 차수를 그대로 반환한다
    (= 제출하지 않고 실험만 반복하면 차수가 바뀌지 않음).
    """
    state = load_version_state()
    if state["submitted"]:
        state["version"] += 1
        state["submitted"] = False
        save_version_state(state)
    return state["version"]

state = load_version_state()
print("현재 차수:", state["version"], "차  (제출 완료 여부:", state["submitted"], ")")

현재 차수: 1 차  (제출 완료 여부: False )


**작동 방식**
- `version_state.json`에 현재 차수가 저장되어, 노트북을 새로 열어도 차수가 유지됩니다.
- "제출 파일 생성" 셀에서 `get_active_version()`을 호출하는 순간: 이전에 제출 완료(`submitted=True`) 상태였다면 차수를 자동으로 1 올리고, 아니면 같은 차수를 유지합니다.
- 즉, 같은 차수 안에서 실험을 몇 번 반복해도 파일명이 바뀌지 않고, **실제로 대회에 제출했다고 표시한 다음부터만** 차수가 올라갑니다.


## 1. 데이터 로드

In [2]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings("ignore")

DATA_DIR = "data/"
TARGET_COL = "임신 성공 여부"
ID_COL = "ID"
RANDOM_STATE = 1004
N_SPLITS = 5

train = pd.read_csv(DATA_DIR + "train.csv")
test = pd.read_csv(DATA_DIR + "test.csv")
print(f"train: {train.shape}, test: {test.shape}")

train: (256351, 69), test: (90067, 68)


## 2. 전처리 (main.py와 동일한 파이프라인)

In [3]:
def drop_id(train, test):
    test_ids = test[ID_COL].copy()
    train = train.drop(columns=[ID_COL])
    test = test.drop(columns=[ID_COL])
    return train, test, test_ids


def add_structural_flags(train, test):
    """DI 시술 전체 결측 / 동결여부 조건부 결측을 압축 플래그로 명시."""
    for df in (train, test):
        df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
        df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return train, test


def drop_uninformative_columns(train, test):
    """정보량이 거의 없는 컬럼 제거 (분산 0에 가까움)."""
    cols_to_drop = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
    train = train.drop(columns=cols_to_drop)
    test = test.drop(columns=cols_to_drop)
    return train, test


def fill_missing_values(train, test):
    """수치형은 -1, 범주형은 '해당없음'으로 명시적 결측 처리."""
    categorical_cols = train.select_dtypes(include=["object", "str"]).columns
    numerical_cols = train.select_dtypes(include=["int64", "float64"]).columns
    na_cols = train.isna().sum().loc[lambda x: x > 0].index

    categorical_na_cols = [c for c in na_cols if c in categorical_cols]
    numerical_na_cols = [c for c in na_cols if c in numerical_cols]

    for col in categorical_na_cols:
        train[col] = train[col].fillna("해당없음")
        test[col] = test[col].fillna("해당없음")
    for col in numerical_na_cols:
        train[col] = train[col].fillna(-1)
        test[col] = test[col].fillna(-1)
    return train, test


def encode_categorical(train, test):
    categorical_cols = train.select_dtypes(include=["object", "str"]).columns
    for col in categorical_cols:
        le = LabelEncoder()
        le.fit(train[col])
        unseen_labels = set(test[col].unique()) - set(le.classes_)
        if unseen_labels:
            le.classes_ = np.append(le.classes_, list(unseen_labels))
        train[col] = le.transform(train[col])
        test[col] = le.transform(test[col])
    return train, test


def preprocess(train, test):
    train, test, test_ids = drop_id(train, test)
    train, test = add_structural_flags(train, test)
    train, test = drop_uninformative_columns(train, test)
    train, test = fill_missing_values(train, test)
    train, test = encode_categorical(train, test)
    return train, test, test_ids


train_p, test_p, test_ids = preprocess(train.copy(), test.copy())
print(f"전처리 완료: {train_p.shape}, {test_p.shape}")

전처리 완료: (256351, 68), (90067, 67)


## 3. 모델 학습 (Stratified 5-Fold 앙상블)

In [4]:
MODEL_NAME = "LGBM"  # 파일명에 들어갈 모델 이름. 다른 모델로 실험하면 이 값을 바꿔서 사용.

X = train_p.drop(columns=[TARGET_COL])
y = train_p[TARGET_COL]

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

models = []
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)
    model.fit(X_train, y_train)

    val_proba = model.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, val_proba)
    fold_scores.append(score)
    models.append(model)
    print(f"[Fold {fold}/{N_SPLITS}] ROC-AUC: {score:.4f}")

best_local_score = np.mean(fold_scores)
print(f"\n5-Fold 평균 ROC-AUC: {best_local_score:.4f} (std: {np.std(fold_scores):.4f})")

[Fold 1/5] ROC-AUC: 0.7434


[Fold 2/5] ROC-AUC: 0.7391


[Fold 3/5] ROC-AUC: 0.7374


[Fold 4/5] ROC-AUC: 0.7382


[Fold 5/5] ROC-AUC: 0.7383

5-Fold 평균 ROC-AUC: 0.7393 (std: 0.0021)


## 4. 테스트 예측

In [5]:
final_pred = np.mean([model.predict_proba(test_p)[:, 1] for model in models], axis=0)
print("예측 완료:", final_pred.shape)

예측 완료: (90067,)


## 5. 제출 파일 생성

파일명 형식: `{N차}_{모델이름}_submit_improved({로컬점수}->{실제점수}).csv`

- `get_active_version()`이 차수를 결정합니다 (이전 제출 완료 상태였으면 +1, 아니면 그대로).
- 아직 실제 점수를 모르므로 `->` 뒤는 빈 값으로 저장됩니다.
- 실제 대회에 제출해서 점수를 받으면, 6번 셀에서 그 점수를 기록하고 제출 완료 처리를 합니다.


In [6]:
VERSION = get_active_version()

LOCAL_SCORE = f"{best_local_score:.5f}"
SAVE_NAME = f"{VERSION}차_{MODEL_NAME}_submit_improved({LOCAL_SCORE}->).csv"
SAVE_PATH = f"data/{SAVE_NAME}"

submission = pd.read_csv(DATA_DIR + "sample_submission.csv")
submission["ID"] = test_ids.values
submission["probability"] = final_pred

submission.to_csv(SAVE_PATH, index=False)
print(f"저장 완료: {SAVE_NAME}")
print(submission.head())

저장 완료: 1차_LGBM_submit_improved(0.73931->).csv
           ID  probability
0  TEST_00000     0.001201
1  TEST_00001     0.002337
2  TEST_00002     0.155487
3  TEST_00003     0.100129
4  TEST_00004     0.501346


## 6. 실제 대회에 제출한 뒤 (점수 확인 후 실행)

**대회 사이트에 위에서 만든 csv를 업로드해서 실제 점수를 받은 뒤에만** 이 셀을 실행하세요.

- `ACTUAL_SCORE`에 받은 실제 점수를 입력
- 이 셀을 실행하면: 방금 만든 파일 이름이 실제 점수가 채워진 이름으로 변경되고, 차수 상태가 "제출 완료"로 바뀜
- 그 다음 노트북을 다시 실행하면 `get_active_version()`이 자동으로 차수를 1 올린 새 차수를 반환합니다.


In [ ]:
ACTUAL_SCORE = "0.0000"  # 대회에서 받은 실제 점수를 여기에 입력하세요

import os

old_path = SAVE_PATH
new_name = f"{VERSION}차_{MODEL_NAME}_submit_improved({LOCAL_SCORE}->{ACTUAL_SCORE}).csv"
new_path = f"data/{new_name}"

os.rename(old_path, new_path)
print(f"파일명 변경 완료: {new_name}")

state = load_version_state()
state["submitted"] = True
save_version_state(state)
print(f"{VERSION}차 제출 완료로 표시됨. 다음 실행부터는 {VERSION + 1}차로 시작됩니다.")